# Tomographic indexing notebook  
__Written by Haixing Fang, Jon Wright and James Ball__  
__Date: 21/02/2025__

This notebook will try to find good grain orientations from a 4D merge of your peak data.  
This means your peaks have been merged across omega and dty.  
This notebook (and the tomo route in general) works best for low levels of deformation.  
If it doesn't seem to work well, try the point-by-point route instead!

In [ ]:
# this cell is tagged with 'parameters'
# to view the tag, select the cell, then find the settings gear icon (right or left sidebar) and look for Cell Tags

# python environment stuff
IMAGED11_PATH = None  # means do not use git, otherwise "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = None  # None means guess, or you can specify a folder for the checkout

# dataset file to import
dset_path = 'si_cube_test/processed/Si_cube/Si_cube_S3DXRD_nt_moves_dty/Si_cube_S3DXRD_nt_moves_dty_dataset.h5'

# which phase to index
phase_str = 'Si'

# peak filtration parameters
min_frames_per_peak = 0
cf_strong_frac = 0.9939
cf_strong_dsmax = 1.594
cf_strong_dstol = 0.005

# indexing parameters
# we have to choose which rings we want to generate orientations on
# generally we want two or three low-multiplicity rings that are isolated from other phases
# take a look at the ring assignment output from a few cells above, and choose two or three
rings_for_gen = [0, 1, 3]
# now we want to decide which rings to score our found orientations against
# generally we can just exclude dodgy rings (close to other phases, only a few peaks in etc)
rings_for_scoring = [0, 1, 2, 3, 4]
# the sequence of hkl tolerances the indexer will iterate through
hkl_tols_seq = [0.01, 0.02, 0.03, 0.04]
# the sequence of minpks fractions the indexer will iterate through
fracs = [0.9, 0.7]
# the max number of UBIs we can find per pair of rings
max_grains = 1000

peak_assign_tol = 0.05

# Optional back/forward projection feedback for peak selection
refine_all_grain_peaks = False
plot_refinement_report = True
refinement_report_limit = 4    # cap detailed previews in batch runs
peak_refinement_options = dict(
    iterations=10,
    mask_level=0.12,
    dilation=2,
    roi_radius=None,
    support_level=0.02,
    beam_sigma_steps=0.75,
    damping=0.5,
    weight_tolerance=0.005,
    minimum_exclusive=5,
    filter_name="hann",
    verbose=True,              # progress bar and refinement statistics
)

# Optional spatial splits and already indexed twin orientations
grain_ids_to_decouple = None    # screen all; [] skips; [0, 2] selects parents
twin_pairs_to_decouple = []     # e.g. [(3, 7)]
plot_decoupling = False        # enable reconstruction plots for inspection
decoupling_options = dict(
    y0=None,                   # calibrated rotation-axis dty if available
    width=None,                # centreline fit tolerance; default 2 dty steps
    width_2d=None,              # learn branch envelopes automatically
    margin=None,
    angle_bin=10,
    density_fraction=0.08,
    trials=600,
    seed=42,
    max_components=8,          # upper search limit, not the requested count
)

# EXPERTS: Can specify par_file as a parameter if you want
par_file = None

In [ ]:
if IMAGED11_PATH is not None:
    exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())
    PYTHONPATH=setup_ImageD11_from_git(CHECKOUT_PATH, IMAGED11_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import display

# When running this notebook from the checkout, use its updated sinogram.py.
# Restart the kernel if ImageD11 has already been imported from another install.
from pathlib import Path
import sys
if IMAGED11_PATH is None:
    local_candidates = [Path.cwd(), Path.cwd() / 'ImageD11', *Path.cwd().parents]
    for candidate in local_candidates:
        if (candidate / 'ImageD11' / 'nbGui' / 'S3DXRD' / 'tomo_1_index.ipynb').is_file():
            loaded = sys.modules.get('ImageD11')
            if loaded is not None and Path(loaded.__file__).resolve().parent != candidate / 'ImageD11':
                raise RuntimeError('Restart this notebook kernel to use the modified ImageD11 checkout')
            sys.path.insert(0, str(candidate))
            break

import ImageD11.grain
import ImageD11.indexing
import ImageD11.columnfile
from ImageD11.peakselect import select_ring_peaks_by_intensity
import ImageD11.nbGui.nb_utils as utils

%matplotlib ipympl

# Load data
## Dataset

In [ ]:
ds = ImageD11.sinograms.dataset.load(dset_path)
print(ds)

## Parameters
Specify the path to your parameter file.

You can optionally set up some default parameters for either an Eiger or Frelon detector like so:
```python
from ImageD11.parameters import AnalysisSchema
asc = AnalysisSchema.from_default(detector='eiger')  # or detector='frelon'
asc.save('./pars.json')
```
Please note in this case that you will still have to update the `geometry.par` values accordingly for your experiment.  
If you haven't already, you should run one of the calibration notebooks to determine these.

In [ ]:
if par_file is not None:
    # only change if ds has no parfile
    if not hasattr(ds, 'parfile') or ds.parfile is None:
        ds.parfile = par_file
        ds.save()

## Phases
If the parameter file was a json, we can access the unit cells via `ds.phases.unitcells`

In [ ]:
ds.phases = ds.get_phases_from_disk()
ds.phases.unitcells

In [ ]:
ucell = ds.phases.unitcells[phase_str]
print(ucell)

## Peaks

In [ ]:
cf_4d = ds.get_cf_4d()
ds.update_colfile_pars(cf_4d)
if not os.path.exists(ds.col4dfile):
    # save the 4D peaks to file so we don't have to spatially correct them again
    ImageD11.columnfile.colfile_to_hdf(cf_4d, ds.col4dfile)

In [ ]:
# Optionally remove some noisy peaks
if min_frames_per_peak > 0:
    cf_4d.filter(cf_4d['npk2d'] > min_frames_per_peak)

# Visualise data
### $d^{*}$ vs $\eta$

In [ ]:
skip = 1  # we can skip peaks to speed up plotting if needed
ucell.makerings(cf_4d.ds.max())
fig, ax = plt.subplots(figsize=(10,5), layout='constrained')
ax.plot(cf_4d.ds[::skip], cf_4d.eta[::skip], ',')
ax.vlines(ucell.ringds, -50, 50, color='red')
ax.set(xlabel=r'$d^{*}~(\AA^{-1})$', ylabel=r'$\eta~(\degree)$', title='2D azimuthal transform')
plt.show()

### Sinogram

In [ ]:
fig, ax = plt.subplots(layout='constrained')
im, om_edges, dty_edges = ds.sinohist(np.log(ds.pk2d['sum_intensity']), ds.pk2d['omega'], ds.pk2d['dty'], return_edges=True)
pcm = ax.pcolormesh(om_edges, dty_edges, im.T)
ax.set(xlabel=r'$\omega~(\degree)$', ylabel='dty', title='Sinogram of all peaks')
cax = fig.colorbar(pcm, ax=ax, label='log(intensity)')
plt.show()

# Filtration
Here we are filtering our peaks (`cf_4d`) to select only the strongest ones for indexing purposes only!  
We first filter the peaks in $d^{*}$ to keep only those close to the predicted peaks from the unit cell.  
We then sort our peaks by intensity, and take a certain intensity-weighted fraction of them.  
`dstol`: The tolerance in $d^{*}$ between a peak and a predicted reflection.  
`dsmax`: The maximum allowed peak $d^{*}$ value. Used to limit the number of rings given to the indexer - 6-8 rings max are normally sufficient.   
`frac`: The intensity fraction: `frac=0.9` keeps 90% of the peak intensity. We recommend that you choose a value close to the 'elbow' of the plot.

In [ ]:
cf_strong = select_ring_peaks_by_intensity(cf_4d, frac=cf_strong_frac, dsmax=cf_strong_dsmax, dstol=cf_strong_dstol, ucell=ucell, doplot=0.5)

In [ ]:
skip = 1  # we can skip peaks to speed up plotting if needed
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.plot(cf_4d.ds[::skip], cf_4d.sum_intensity[::skip],',', label='cf_4d')
ax.plot(cf_strong.ds[::skip], cf_strong.sum_intensity[::skip],',', label='cf_strong')
ax.vlines(ucell.ringds, 1e3, 1e4, color='red')
ax.set(xlabel=r'$d^{*}~(\AA^{-1})$', ylabel='Intensity', yscale='log', title='Peak filtration')
ax.legend()
plt.show()

# Indexing
## Ring assignment

In [ ]:
indexer = ImageD11.indexing.indexer_from_colfile_and_ucell(cf_strong, ucell)
indexer.ds_tol = cf_strong_dstol
ImageD11.indexing.loglevel = 1
indexer.assigntorings()
ImageD11.indexing.loglevel = 3
print(f"Indexing {cf_strong.nrows} peaks")

In [ ]:
skip = 1  # we can skip peaks to speed up plotting if needed
fig, ax = plt.subplots(layout='constrained', figsize=(10,5))
ax.scatter(indexer.colfile.ds[::skip], indexer.colfile.eta[::skip], c=indexer.ra[::skip]%20, cmap='tab20', s=1)
ax.vlines(ucell.ringds, -50, 50, color='red')
ax.set(xlabel=r'$d^{*}~(\AA^{-1})$', ylabel=r'$\eta~(\degree)$', xlim=(min(ucell.ringds[0], cf_strong.ds.min()) - 0.02, cf_strong.ds.max() + 0.02), title='Ring assignments')
plt.show()

In [ ]:
cosine_tol = np.cos(np.radians(90 - ds.ostep))
grains, indexer = utils.do_index(cf=cf_strong,
                                 unitcell=ds.phases.unitcells[phase_str],
                                 dstol=cf_strong_dstol,
                                 forgen=rings_for_gen,
                                 foridx=rings_for_scoring,
                                 hkl_tols=hkl_tols_seq,
                                 fracs=fracs,
                                 cosine_tol=cosine_tol,
                                 max_grains=max_grains,
)
for ginc, g in enumerate(grains):
    g.gid = ginc
print(f'Found {len(grains)} grains!')
# Stable inputs when the decoupling cell is rerun.
grains_before_decoupling = list(grains)


## Results

In [ ]:
unit_cell_lengths = np.array([np.sort(g.unitcell.copy()) for g in grains])
median_unit_cell = np.median(unit_cell_lengths, axis=0)
print("Median unit cell:", median_unit_cell)
fig, ax = plt.subplots(constrained_layout=True)
ax.plot(unit_cell_lengths[:,:3])
ax.hlines(median_unit_cell[:3], 0, len(unit_cell_lengths))
ax.set(xlabel='Grain ID', ylabel='Unit cell length')
plt.show()

In [ ]:
utils.assign_peaks_to_grains(grains, cf_strong, tol=peak_assign_tol)
cf_strong_before_decoupling = cf_strong.copy()

In [ ]:
utils.plot_index_results(indexer, cf_strong, 'Indexing attempt')

In [ ]:
utils.plot_grain_sinograms(grains, cf_strong, min(len(grains), 25))

## Optional spatial decoupling and known twin pairs
Set `grain_ids_to_decouple=None` to screen every indexed grain in batch mode. Only unambiguous multi-track parents below `max_components` are decoupled; single-track parents retain their original grains and peaks. Ambiguous counts, counts at the search limit, and per-parent failures retain the original grain and are listed in `decoupling_report`. Use `plot_decoupling=True` to inspect split reconstructions. A list such as `[0, 2]` restricts processing to those parents and keeps the existing fail-fast behavior. Explicit mappings such as `{0: 3}` remain available as overrides. Child 0 keeps the parent's ID; the other children are appended. For two already-indexed twin-related orientations, use `twin_pairs_to_decouple` instead; both existing IDs and distinct UBIs are retained. Explicit entries must be disjoint. In all-grain mode, known twin-pair members are excluded from spatial screening. Empty lists skip spatial screening.

Inspect the split reconstructions before the export cell. The exported fractional assignments are reused by `GrainSinogram.prepare_peaks_from_4d` in the mapping notebook; avoid replacing the split with orientation-only assignments. Twin-exclusive reflections seed each orientation, and common reflections are shared according to projected density. The twin law and missing orientations are not inferred. Use the modified ImageD11 checkout in the notebook kernel.


In [ ]:
from ImageD11.sinograms.sinogram import (
    decouple_grain_sinograms,
    refine_grain_sinograms,
    save_decoupled_grains,
    save_refined_grains,
)

# Specialized fitting controls override the common feedback controls.
decoupling_parameters = peak_refinement_options.copy()
decoupling_parameters.update(decoupling_options)
decoupling_parameters.setdefault("hkl_tol", peak_assign_tol)
decoupling_report = []
grains, decoupling_results, cf_strong_decoupled = decouple_grain_sinograms(
    ds,
    grains_before_decoupling,
    cf_4d,
    grain_ids=grain_ids_to_decouple,
    twin_pairs=twin_pairs_to_decouple,
    cf_strong=cf_strong_before_decoupling,
    report=decoupling_report,
    **decoupling_parameters,
)
grains_before_peak_refinement = list(grains)
if plot_decoupling:
    utils.plot_decoupled_sinograms(decoupling_results)
    plt.show()
for result_index, result in enumerate(decoupling_results):
    print(f"Grains {result.parent_ids} -> {result.child_ids}; converged={result.converged}")
    if "component_selection" in result.stats:
        print("Automatic count:", result.stats["component_selection"])
    refinement_report_figures = utils.report_sinogram_refinement(
        result, max_grains=refinement_report_limit,
        plot=plot_refinement_report and result_index < refinement_report_limit)
    for figure in refinement_report_figures:
        display(figure)
        plt.close(figure)
print({status: sum(row["status"] == status for row in decoupling_report)
       for status in ("split", "unchanged", "review", "failed")})
for row in decoupling_report:
    if row["status"] in ("review", "failed"):
        print(row)
print(f"{len(grains)} grains ready for export")


## Optional joint peak refinement for all grains
Set `refine_all_grain_peaks=True` and adjust `peak_refinement_options` in the parameter cell. This improves peak selection while retaining the current grain count and UBIs. Spatial/twin results seed the shared projection loop through `initial_results`; empty decoupling lists permit ordinary refinement of all indexed grains. Keep the original peak table for parameter trials and inspect the masks and convergence before exporting.

All grains use a common rotation-axis offset. A calibrated `y0` in `decoupling_options` is also passed to joint refinement; otherwise refinement estimates one common median axis.


In [ ]:
# Start from the same grains and original peaks on every parameter trial.
grains = list(grains_before_peak_refinement)
peak_refinement_result = None
cf_strong_refined = None
if refine_all_grain_peaks:
    refinement_parameters = peak_refinement_options.copy()
    refinement_parameters.setdefault("y0", decoupling_parameters.get("y0"))
    refinement_parameters.setdefault("use_saved_assignments", False)
    grains, peak_refinement_result, cf_strong_refined = refine_grain_sinograms(
        ds,
        grains_before_peak_refinement,
        cf_4d,
        cf_strong=cf_strong_before_decoupling,
        initial_results=decoupling_results,
        phase_name=phase_str,
        hkl_tol=peak_assign_tol,
        **refinement_parameters,
    )
    utils.plot_grainsino_grid(peak_refinement_result.grainsinos)
    plt.show()
    print("Peak refinement converged:", peak_refinement_result.converged)
    refinement_report_figures = utils.report_sinogram_refinement(
        peak_refinement_result, max_grains=refinement_report_limit,
        plot=plot_refinement_report)
    for figure in refinement_report_figures:
        display(figure)
        plt.close(figure)


# Export data

In [ ]:
if peak_refinement_result is not None:
    backup = save_refined_grains(ds, grains, peak_refinement_result, phase_name=phase_str)
else:
    backup = save_decoupled_grains(ds, grains, decoupling_results, phase_name=phase_str)
if backup is not None:
    print('Previous grain file backed up to:', backup)
ds.save()
